# DeFoG — Colab Environment Setup (run once)

This notebook builds the `defog` conda environment **directly on the Colab GPU instance**
and saves it to Google Drive as a compressed tarball for fast restore in future sessions.

**Run this notebook once (or whenever dependencies change).**  
For every subsequent session, open and run `02_session_init.ipynb` instead.

### Before running
1. Set the runtime to **GPU** (Runtime → Change runtime type → T4/A100).
2. Set `GITHUB_REPO` in cell 4 to your fork's URL.
3. Upload the two precomputed `.pt` files to Drive **after** cell 2 completes:
   - `MyDrive/DeFoGColab/data/qm9/qm9_motif_marginals_6x1.pt`
   - `MyDrive/DeFoGColab/data/qm9/spminer_motif_marginals.pt`

### Expected total runtime
~25–35 min (graph-tool conda solve: ~10 min, conda-pack: ~8 min, Drive upload: ~5 min).

In [1]:
# ── Cell 1: Mount Drive and create folder structure ───────────────────────────
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/DeFoGColab'
for d in [
    f'{DRIVE_BASE}',
    f'{DRIVE_BASE}/data/qm9',
    f'{DRIVE_BASE}/checkpoints',
]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted.')
print(f'DeFoGColab root: {DRIVE_BASE}')
print('Folder structure created.')

Mounted at /content/drive
Drive mounted.
DeFoGColab root: /content/drive/MyDrive/DeFoGColab
Folder structure created.


In [2]:
# ── Cell 2: Verify GPU and CUDA ───────────────────────────────────────────────
# The CUDA version shown here determines which PyTorch wheel will be installed.
# colab_create_env.sh detects this automatically.
!nvidia-smi

Sat Apr 25 02:16:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P0             49W /  350W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
%%bash
# ── Cell 3: Install Miniforge ─────────────────────────────────────────────────
# Miniforge is the conda-forge project's own conda installer.
# It ships pre-configured to use conda-forge only, so no Anaconda
# Terms-of-Service acceptance is ever required (unlike Miniconda ≥ 2024).
set -e
if [ -f /content/miniconda3/bin/conda ]; then
    echo "Conda already installed, skipping."
    /content/miniconda3/bin/conda --version
    exit 0
fi
echo "Downloading Miniforge..."
wget -q \
  https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh \
  -O /tmp/miniforge.sh
bash /tmp/miniforge.sh -b -p /content/miniconda3
rm /tmp/miniforge.sh
echo "Miniforge installed."
/content/miniconda3/bin/conda --version

PREFIX=/content/miniconda3
Unpacking bootstrapper...
Unpacking payload...
Extracting ca-certificates-2026.2.25-hbd8a1cb_0.conda
Extracting libgomp-15.2.0-he0feb66_18.conda
Extracting libzlib-1.3.2-h25fd6f3_2.conda
Extracting nlohmann_json-abi-3.12.0-h0f90c79_1.conda
Extracting pybind11-abi-11-hc364b38_1.conda
Extracting python_abi-3.13-8_cp313.conda
Extracting tzdata-2025c-hc9c84f9_1.conda
Extracting _openmp_mutex-4.5-20_gnu.conda
Extracting zstd-1.5.7-hb78ec9c_6.conda
Extracting ld_impl_linux-64-2.45.1-default_hbd61a6d_101.conda
Extracting libgcc-15.2.0-he0feb66_18.conda
Extracting bzip2-1.0.8-hda65f42_9.conda
Extracting c-ares-1.34.6-hb03c661_0.conda
Extracting keyutils-1.6.3-hb9d3cd8_0.conda
Extracting libexpat-2.7.4-hecca717_0.conda
Extracting libffi-3.5.2-h3435931_0.conda
Extracting libgcc-ng-15.2.0-h69a702a_18.conda
Extracting libiconv-1.18-h3b78370_2.conda
Extracting liblzma-5.8.2-hb03c661_0.conda
Extracting libmpdec-4.0.0-hb03c661_1.conda
Extracting libstdcxx-15.2.0-h934c35e_18

In [5]:
# ── Cell 4: Configure — set your GitHub repo URL here ─────────────────────────
GITHUB_REPO = 'https://github.com/Maxlyu254/DeFoGPlus.git'
REPO_DIR    = '/content/DeFoGPlus'

print(f'Will clone: {GITHUB_REPO}')
print(f'       to : {REPO_DIR}')

Will clone: https://github.com/Maxlyu254/DeFoGPlus.git
       to : /content/DeFoGPlus


In [6]:
%%bash -s "$GITHUB_REPO" "$REPO_DIR"
# ── Cell 5: Clone repo ────────────────────────────────────────────────────────
GITHUB_REPO="$1"
REPO_DIR="$2"
set -e
if [ -d "$REPO_DIR/.git" ]; then
    echo "Repo already present, pulling latest."
    git -C "$REPO_DIR" pull
else
    git clone "$GITHUB_REPO" "$REPO_DIR"
fi
echo "Repo ready at $REPO_DIR"
ls "$REPO_DIR"

Repo ready at /content/DeFoGPlus
colab
configs
docker
Dockerfile
environment.yaml
images
LICENSE
README.md
requirements.txt
scripts
setup.py
shell
src


Cloning into '/content/DeFoGPlus'...


In [7]:
%%bash
# ── Cell 6: Create defog conda environment ────────────────────────────────────
# This is the longest step (~15-20 min).
# graph-tool from conda-forge takes ~10 min to solve and download.
set -e
bash /content/DeFoGPlus/shell/colab_create_env.sh 2>&1

Detected CUDA 12.8
PyTorch wheel index: https://download.pytorch.org/whl/cu124

=== Step 1: Creating conda environment (Python 3.9) ===
Retrieving notices: - \ | done
Channels:
 - conda-forge
Platform: linux-64
Solving environment: - done


==> WARNING: A newer version of conda exists. <==
    current version: 26.1.1
    latest version: 26.3.2

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /content/miniconda3/envs/defog

  added / updated specs:
    - python=3.9


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ca-certificates-2026.4.22  |       hbd8a1cb_0         128 KB  conda-forge
    ld_impl_linux-64-2.45.1    |default_hbd61a6d_102         711 KB  conda-forge
    libexpat-2.7.5             |       hecca717_0          75 KB  conda-forge
    liblzma-5.8.3              |       hb03c661_0         111

In [8]:
# ── Cell 7: Smoke test — verify key imports ───────────────────────────────────
import subprocess, os

PYTHON = '/content/miniconda3/envs/defog/bin/python'

check = """
import graph_tool
import torch
import rdkit
import torch_geometric
import pytorch_lightning
import hydra
import wandb
print('graph_tool       :', graph_tool.__version__)
print('torch            :', torch.__version__)
print('CUDA available   :', torch.cuda.is_available())
print('torch_geometric  :', torch_geometric.__version__)
print('pytorch_lightning:', pytorch_lightning.__version__)
print('All imports OK.')
"""

# Set DISPLAY so graph_tool does not attempt to open an X window.
env = {**os.environ, 'MPLBACKEND': 'Agg'}
r = subprocess.run([PYTHON, '-c', check], capture_output=True, text=True, env=env)
print(r.stdout)
if r.returncode != 0:
    print('STDERR:', r.stderr)
    raise RuntimeError('Smoke test failed — check the output above.')

graph_tool       : 2.45 (commit b1a649d8, )
torch            : 2.4.0+cu124
CUDA available   : True
torch_geometric  : 2.3.1
pytorch_lightning: 2.0.4
All imports OK.



In [9]:
%%bash
# ── Cell 8: Pack environment to Drive ────────────────────────────────────────
# conda-pack creates a self-contained tarball of the entire environment,
# including both conda and pip packages. conda-unpack (run at restore time)
# rewrites any baked-in absolute paths.
#
# Expected tarball size: 2-4 GB compressed.
# Expected runtime: 5-10 min (packing) + 3-8 min (Drive upload).
set -e
TARBALL_TMP='/tmp/defog_env.tar.gz'
TARBALL_DST='/content/drive/MyDrive/DeFoGColab/defog_env.tar.gz'

echo "Packing environment — this will take several minutes..."
/content/miniconda3/envs/defog/bin/conda-pack \
    -p /content/miniconda3/envs/defog \
    -o "$TARBALL_TMP" \
    --ignore-missing-files

echo "Pack complete. Size: $(du -sh $TARBALL_TMP | cut -f1)"
echo "Copying to Drive..."
cp "$TARBALL_TMP" "$TARBALL_DST"
rm "$TARBALL_TMP"
echo "Uploaded: $(du -sh $TARBALL_DST | cut -f1)  →  $TARBALL_DST"

Packing environment — this will take several minutes...


/content/miniconda3/envs/defog/lib/python3.9/site-packages/conda_pack/core.py:16: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
CondaPackError: Failed to determine path to environment: 'defog'. This may be due to conda not being on your PATH. The full error is below:

b''


CalledProcessError: Command 'b'# \xe2\x94\x80\xe2\x94\x80 Cell 8: Pack environment to Drive \xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\n# conda-pack creates a self-contained tarball of the entire environment,\n# including both conda and pip packages. conda-unpack (run at restore time)\n# rewrites any baked-in absolute paths.\n#\n# Expected tarball size: 2-4 GB compressed.\n# Expected runtime: 5-10 min (packing) + 3-8 min (Drive upload).\nset -e\nTARBALL_TMP=\'/tmp/defog_env.tar.gz\'\nTARBALL_DST=\'/content/drive/MyDrive/DeFoGColab/defog_env.tar.gz\'\n\nech

## ✅ Setup complete

The environment tarball is now saved at:
```
MyDrive/DeFoGColab/defog_env.tar.gz
```

### Next: upload the precomputed `.pt` files to Drive

In the Drive web UI, upload these two files to `MyDrive/DeFoGColab/data/qm9/`:
- `qm9_motif_marginals_6x1.pt`  (ring-edited marginals)
- `spminer_motif_marginals.pt`  (SPMiner-mined marginals)

They will be accessible through the `data/` symlink created in every session by `02_session_init.ipynb`.

From now on, start each new Colab session with **`02_session_init.ipynb`**.